In [2]:
from pathlib import Path
from ib_async import IB, util, Stock, Index

# In notebooks
util.startLoop()

ib = IB()
ib.connect('127.0.0.1', 7496, clientId=5, timeout=5)
print('connected:', ib.isConnected())

# Real-time if available; use 3 or 4 if you want delayed/frozen as fallback
ib.reqMarketDataType(1)

# -----------------------------
# Asset pool
# -----------------------------
stock_tickers = [
    # 11 sector ETFs
    'XLB', 'XLE', 'XLF', 'XLK', 'XLI', 'XLP',
    'XLU', 'XLV', 'XLY', 'XLRE', 'XLC',
    
    # useful additions
    'QQQ', 'IWM', 'TLT', 'USO', 'GLD',
    
    # optional benchmark / credit ETFs
    'SPY', 'HYG', 'LQD'
]

# VIX is an index, not a stock
index_tickers = ['VIX']

# Save beside notebook
save_dir = Path('.')

# -----------------------------
# Fetch stock/ETF history
# -----------------------------
for ticker in stock_tickers:
    try:
        contract = Stock(ticker, 'SMART', 'USD')
        ib.qualifyContracts(contract)

        bars = ib.reqHistoricalData(
            contract,
            endDateTime='',
            durationStr='26 Y',
            barSizeSetting='1 day',
            whatToShow='TRADES',
            useRTH=True,
            formatDate=1
        )

        df = util.df(bars)
        df.to_csv(save_dir / f'{ticker}.csv', index=False)
        print(f'Saved {ticker}.csv with {len(df)} rows')

    except Exception as e:
        print(f'Failed for {ticker}: {e}')

# -----------------------------
# Fetch index history (VIX)
# -----------------------------
for ticker in index_tickers:
    try:
        contract = Index(ticker, 'CBOE', 'USD')
        ib.qualifyContracts(contract)

        bars = ib.reqHistoricalData(
            contract,
            endDateTime='',
            durationStr='26 Y',
            barSizeSetting='1 day',
            whatToShow='TRADES',   # if this fails, try 'MIDPOINT'
            useRTH=True,
            formatDate=1
        )

        df = util.df(bars)
        df.to_csv(save_dir / f'{ticker}.csv', index=False)
        print(f'Saved {ticker}.csv with {len(df)} rows')

    except Exception as e:
        print(f'Failed for {ticker}: {e}')

ib.disconnect()
print('done')

connected: True
Saved XLB.csv with 6534 rows
Saved XLE.csv with 6534 rows
Saved XLF.csv with 6534 rows
Saved XLK.csv with 6534 rows
Saved XLI.csv with 6534 rows
Saved XLP.csv with 6534 rows
Saved XLU.csv with 6534 rows
Saved XLV.csv with 6534 rows
Saved XLY.csv with 6534 rows
Saved XLRE.csv with 2627 rows
Saved XLC.csv with 1949 rows
Saved QQQ.csv with 6536 rows
Saved IWM.csv with 6490 rows
Saved TLT.csv with 2547 rows
Saved USO.csv with 5017 rows
Saved GLD.csv with 5365 rows
Saved SPY.csv with 6533 rows
Saved HYG.csv with 4766 rows
Saved LQD.csv with 5917 rows
Saved VIX.csv with 5131 rows
done


In [3]:
import pandas as pd
from pathlib import Path

tickers = stock_tickers + index_tickers
dfs = []

for ticker in tickers:
    path = Path(f'{ticker}.csv')
    if path.exists():
        df = pd.read_csv(path)
        df['date'] = pd.to_datetime(df['date'])
        df = df[['date', 'close']].rename(columns={'close': ticker})
        dfs.append(df.set_index('date'))

merged = pd.concat(dfs, axis=1).sort_index()
merged.to_csv('merged_close.csv')
print('Saved merged_close.csv')

Saved merged_close.csv
